In [133]:
import re
import os
import pandas as pd
import numpy

In [3]:
fp = os.getcwd()
fp

'/Users/AS/coding/llm_classification'

In [4]:
fp = os.path.dirname(os.getcwd())
fp


'/Users/AS/coding'

In [ ]:
def load_data(file_name = 'Social_Media_Sentiment_Analysis_AI_Trends_2026.csv', usecols=['post_text']) -> pd.DataFrame:
    FILE_PATH = os.path.abspath(__file__)
    SRC_FOLDER = os.path.dirname(FILE_PATH)
    PROJECT_FOLDER = os.path.dirname(SRC_FOLDER)
    DATA_FOLDER = os.path.join(PROJECT_FOLDER, 'data')
    data_path = os.path.join(DATA_FOLDER, file_name)
    data = pd.read_csv(data_path, usecols=usecols)
    return data

data = load_data()

In [ ]:
from src.llm_client import LLMClient
import json

CLASSIFIER_SYSTEM = """
You are comments classifier. Classify the following messages as:
positive
negative
neutral

Return only valid JSON with double quotes and no markdown fence:
{
    "sentiment": "positive" | "negative" | "neutral"
}
"""

client = LLMClient()
results = []
# classify and return data
for text in data['post_text']:
    answ = client.chat(text, sys_prompt=CLASSIFIER_SYSTEM)
    answ = json.loads(answ)
    results.append({
        "post_text":text,
        "sentiment":answ['sentiment']
    })

marked_posts = pd.DataFrame(results)

In [81]:
def batch_text(data: pd.Series):
    """
    Transforms pandas series data into a string formatted for LLM processing
    
    Example input: 
        0 I don’t agree with this at all.
    
    Example output: 
        [ROW ID: 0]
        Text: I don’t agree with this at all.
    """
    blocks = []
    for idx, text in data.items():
        block = f"[ROW ID: {idx}]\nText: {text}"
        blocks.append(block)
    
    return '\n\n'.join(blocks)

In [ ]:
def json_preprocessing(llm_answer, expected_rows_ids):
   """
   Function protects from:
      - markdown fences
      - bad JSON shape
      - header rows
      - empty rows
      - string "row_id" instead of integer row ID
      - random extra rows
      - missing predictions
   """
   # clear markdown fences
   cleaned_text = llm_answer.strip()
   cleaned_text = re.sub("^```json\s*", "", cleaned_text)
   cleaned_text = re.sub("^```\s*", "", cleaned_text)
   cleaned_text = re.sub("\s*```$", "", cleaned_text)

   # parse json
   json_answ = json.loads(cleaned_text)

   # validate rows
   valid_rows = []
   
   for row in json_answ:
      # check if row contains valid dictionary
      if not isinstance(row, dict):
         continue

      # must contain required keys
      if "row_id" not in row or "label" not in row:
         continue

      row_id = row['row_id']
      label = row['label']

      # row_id must be an integer
      if not isinstance(row_id, int):
         continue

      # label is non-empty string
      if not isinstance(label, str) or label.strip() == "":
         continue

      # indexes in batch belongs to the passed data
      if row_id not in expected_rows_ids:
         continue

      valid_rows.append({
         "row_id": row_id,
         "label": label.strip()
      })

      # detect missing rows
      df_validated = pd.DataFrame(valid_rows)
      returned_ids = set(df_validated['row_id']) if not df_validated.empty else set()
      missing_ids = set(expected_rows_ids) - returned_ids

   return valid_rows, missing_ids

In [ ]:
from src.llm_client import LLMClient
import json

CLASSIFIER_SYSTEM = """
You are comments classifier. Classify the following rows as:
positive
negative
neutral

Return ONLY a JSON array.

Each item must contain exactly:
- row_id (integer)
- label (string)

Do NOT:
- include headers
- include column names as data
- include explanations
- include markdown
- include empty rows
- include summary rows
- include examples

Return one object for every input row.
"""

results_short = []
missing_rows = []

client = LLMClient()
batch_size = 10
loop_data = data.loc[:100, 'post_text']

# classify and return data
for n in range(0, len(loop_data), batch_size):
    batch = loop_data[n:n+batch_size]
    batch_string = batch_text(batch)
    answ = client.chat(batch_string, sys_prompt=CLASSIFIER_SYSTEM)
    json_answ, missing_ids = json_preprocessing(answ, )
    results_short.extend(json_answ)
    missing_rows.extend(missing_ids)

# join the initial data frame with the llm classification
marked_posts_short = loop_data.to_frame().reset_index().merge(
                        pd.DataFrame(results_short), 
                        how='left', 
                        left_on='index', 
                        right_on='row_id'
                        )

print("Missing_rows:", missing_rows)
print()
marked_posts_short

,index,post_text,row_id,label
0,0,I don’t agree with this at all.,0,negative
1,1,Worst decision ever made.,1,negative
2,2,This trend is getting out of hand.,2,negative
3,3,Pretty neutral about the update.,3,neutral
4,4,"Loved the explanation, very helpful.",4,positive
...,...,...,...,...
96,96,I don’t agree with this at all.,96,negative
97,97,I don’t agree with this at all.,97,negative
98,98,Worst decision ever made.,98,negative
99,99,This is absolutely amazing!,99,positive
